In [2]:
import httpx
import json
from enum import Enum

BASE_MOVIE_API_URL = "https://nomad-movies-2.nomadcoders.workers.dev"


class Endpoint(str, Enum):
    POPULAR_MOVIES = "/movies"
    MOVIE_DETAILS = "/movies/{id}"
    SIMILAR_MOVIES = "/movies/{id}/similar"
    # MOIVE_CREDITS = "/movies/{id}/credits"


apiClient = httpx.Client(base_url=BASE_MOVIE_API_URL)

def get(endpoint: Endpoint, **path_params):
    path = endpoint.format(**path_params)
    response = apiClient.get(path)
    response.raise_for_status()
    return response.json()

def decode(string):
    try:
        return json.loads(string)
    except json.JSONDecodeError:
        return {}

def encode(data):
    try:
        return json.dumps(data)
    except TypeError:
        return ""
        

In [3]:
def getPopularMovies():
    return get(Endpoint.POPULAR_MOVIES)


def getMovieDetails(id):
    return get(Endpoint.MOVIE_DETAILS, id=id)


def getSimilarMovies(id):
    return get(Endpoint.SIMILAR_MOVIES, id=id)


TOOLS = {
    "getPopularMovies": getPopularMovies,
    "getMovieDetails": getMovieDetails,
    "getSimilarMovies": getSimilarMovies,
}


In [7]:
import openai
from openai.types.chat import ChatCompletionMessage

client = openai.OpenAI()
memories = []
tools = [
    {
        "type": "function",
        "function": {
            "name": "getPopularMovies",
            "description": "인기 영화 목록"
        },
    },
    {
        "type": "function",
        "function": {
            "name": "getMovieDetails",
            "description": "영화 상세 정보 조회",
            "parameters": {
                "type": "object",
                "properties": {"id": {"type": "string", "description": "영화 식별 id"}},
                "required": ["id"]
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "getSimilarMovies",
            "description": "유사한 영화 조회",
            "parameters": {
                "type": "object",
                "properties": {"id": {"type": "string", "description": "영화 식별 id"}},
                "required": ["id"]
            },
        },
    }
]

def processAiResponse(message: ChatCompletionMessage):
    toolCalls = message.tool_calls
    answer = message.content

    # 호출할 도구가 아닌 답변인 경우 early return
    if not toolCalls:
        print(f"AI: {answer}")
        memories.append({"role": "assistant", "content": answer})
        return

    # ai가 선택한 도구 메모리에 추가
    memories.append(
        {
            "role": "assistant",
            "content": message.content or "",
            "tool_calls": [
                {
                    "id": toolCall.id,
                    "type": "function",
                    "function": {
                        "name": toolCall.function.name,
                        "arguments": toolCall.function.arguments,
                    },
                }
                for toolCall in toolCalls
            ],
        }
    )

    # ai가 선택한 도구 모두 실행
    for toolCall in toolCalls:
        functionName = toolCall.function.name
        arguments = decode(toolCall.function.arguments)

        # 도구 호출
        print(f"실행할 함수: {functionName} with {arguments}")
        result = TOOLS.get(functionName)(**arguments)
        print(
            f"실행 완료 {functionName} with args {arguments} for a result of {result}"
        )

        # 결과 메모리에 저장
        memories.append(
            {
                "role": "tool",
                "tool_call_id": toolCall.id,
                "name": functionName,
                "content": encode(result),
            }
        )

    # ai 답변 호출
    getAnswer()


def setMessage(message):
    memories.append({"role": "user", "content": message})


def getAnswer():
    response = client.chat.completions.create(
        model="gpt-4o-mini", messages=memories, tools=tools
    )

    processAiResponse(response.choices[0].message)


while True:
    message = input("입력하세요!")
    print(f"ME: {message}")

    if message == "quit" or message == "q":
        break
    else:
        setMessage(message)
        getAnswer()

ME: 지금 인기 있는 영화 알려줘
실행할 함수: getPopularMovies with {}
실행 완료 getPopularMovies with args {} for a result of [{'adult': False, 'backdrop_path': 'https://image.tmdb.org/t/p/w1280/4k99kV4R1bbbrsnjR205v91Xbin.jpg', 'genre_ids': [27, 53], 'id': 1339713, 'title': 'Obsession', 'original_language': 'en', 'original_title': 'Obsession', 'overview': 'After breaking the mysterious "One Wish Willow" to win his crush\'s heart, a hopeless romantic finds himself getting exactly what he asked for but soon discovers that some desires come at a dark, sinister price.', 'popularity': 796.2753, 'poster_path': 'https://image.tmdb.org/t/p/w780/bRwnj8WEKBCvmfeUNOukJPwB43K.jpg', 'release_date': '2026-05-13', 'softcore': False, 'video': False, 'vote_average': 7.9, 'vote_count': 739}, {'adult': False, 'backdrop_path': 'https://image.tmdb.org/t/p/w1280/oPsRr7AfNLw6XaPuMpvkWK0bIUA.jpg', 'genre_ids': [28, 18], 'id': 1057265, 'title': 'Peddi', 'original_language': 'te', 'original_title': 'పెద్ది', 'overview': 'In 1980s 